# Day 1 — DiD warm-up on simulated residential load

**Goal:** Calibrate causal intuition before real smart-meter data.

1. Why naive before/after fails when weather moves too
2. Recover a **known** peak-hour treatment effect with difference-in-differences
3. Report an honest confidence interval

See also: [`docs/causal_mental_model.md`](../docs/causal_mental_model.md)

In [ ]:
from src.causal.did import estimate_did, naive_before_after
from src.causal.simulate import SimulationConfig, simulate_residential_load, summarize_panel

## Simulate a tariff pilot

- Treated homes receive a peak-shaving incentive **after day 30**
- A **cold snap** (+0.8 kW) hits **both** groups after the tariff date
- True ATT on peak hours: **−0.4 kW**

In [ ]:
config = SimulationConfig(att_kw=-0.4, weather_jump_kw=0.8, seed=7)
panel = simulate_residential_load(config)
summarize_panel(panel)

## Naive before/after (wrong)

Treated homes only, peak hours — confounded by the shared weather shock.

In [ ]:
naive = naive_before_after(panel, peak_hours=config.peak_hours)
print(f"Naive treated before/after: {naive:.3f} kW")
print(f"True ATT: {config.att_kw:.3f} kW")

## Difference-in-differences (recovers ATT)

Model: `load_kw ~ treated + post + treated:post`

In [ ]:
result = estimate_did(panel, peak_hours=config.peak_hours)
print(f"DiD ATT: {result.att_kw:.3f} kW (SE {result.std_error:.3f})")
print(f"95% CI: [{result.ci_low:.3f}, {result.ci_high:.3f}]")
print(f"N obs: {result.n_obs}")

## Tariff translation (toy numbers)

With 50 treated homes and ATT ≈ −0.4 kW on peak hours:

- **Peak MW avoided** ≈ 0.4 × 50 / 1000 ≈ **0.02 MW**
- At **€120/MWh** wholesale, one peak hour saves ≈ **€2.4/h** (illustrative)

Real pilots need the CI above, not the point estimate alone.